## Project #2. Scheduling the NBA ##

The National Basketball Association (NBA) is planning their schedule for the 2025/2026 season. In the attached games.csv you can find a (ficticious) preliminary schedule Among other data, you find, for each match, the home team, the away team, and the date and location of the match. The goal of the project is to possibly improve the current draft of the schedule, under some of the constraints imposed by the current schedule.

The project is composed of 4 questions, all to be solved using Integer Programming, to be completed in this order:

1. Write codes that computes and prints, for each team i, the following information:

(a) all the dates when team i played home; (b) for each team j, the number of times team i played against team j at home; (c) for each team j, the number of times team i played against team j away (i.e., at j's home); (d) all the dates when team j played away.

2. Write an Integer Program (without objective function) whose feasible solutions are all the feasible schedules, where a schedule is feasible if, for each team i:

(e) i plays home (possibly, against a different team) exactly on the dates computed in (a) above; (f) plays away (possibly, against a different team) exactly on the dates computed in (d) above; (g) for each team j, i plays home against team j exactly the number of times computed in (b) above; (h) for each team j, i plays away against team j (i.e., at j's home) exactly the number of times computed in (c) above.

3. Compute a feasible schedule that satisfy the following additional constraint: no team should play three consecutive matches where the sum of the absolute values of the difference between the time zones of two consecutive matches is 4 or more; or conclude that no such schedule exists. For instance, if a team plays game 1,2,3 and:
- the time zone difference between the arena where game 1 is played and the arena where game 2 is played is 2;
- the time zone difference between the arena where game 2 is played and the arena where game 3 is played is 3;
Then the schedue is infeasible, since 3+2 = 5 $\geq$ 4.

4. **This part will not be graded, but we may ask you about it in the one-on-one discussion** Compute any improvement to the current schedule. For instance, using Google Maps, you could compute and store the locations of all arenas, compute a feasible schedule that minimizes the maximum distance traveled by a team, and compare this value with the one of the current schedule to show the improvement. You can also access the full 2023/2024 season schedule at https://www.basketball-reference.com/leagues/NBA_2024_games-october.html.

**Deliverables**
- ipynb file containing all the code
- pdf file explaining the models, the variable, and the constraints
- Schedules saved in a .csv or analagous file, presenting the list of matches organized as follows: (a) date (b) team playing home (c) team playing away (d)


## Problem 1: Extracting Structural Scheduling Constraints ##

Write codes that computes and prints, for each team i, the following information:

(a) all the dates when team i played home; 
(b) for each team j, the number of times team i played against team j at home; 
(c) for each team j, the number of times team i played against team j away (i.e., at j's home); 
(d) all the dates when team j played away.

In [19]:
import pandas as pd
from collections import defaultdict

In [21]:
print("\n------------------------------------------------------------")
print("Problem 1: Extracting Structural Scheduling Constraints")
print("------------------------------------------------------------\n")

# -----------------------------
# Load the schedule dataset
# -----------------------------
path = "games.csv"   # Change this path if needed
games = pd.read_csv(path)

# Rename columns for convenience
games = games.rename(columns={
    "Visitor": "AwayTeam",
    "Home": "HomeTeam",
    "Date": "Date"
})

games["Date"] = pd.to_datetime(games["Date"])

# Convert date to datetime format
games["Date"] = pd.to_datetime(games["Date"])

# List of all teams
teams = sorted(set(games["HomeTeam"]).union(set(games["AwayTeam"])))
print("Total number of teams:", len(teams))

# -------------------------------------------------------
# For each team i, print:
# (a) all home dates
# (b) number of times i plays HOME vs each team j
# (c) number of times i plays AWAY vs each team j
# (d) all away dates
# -------------------------------------------------------

for team in teams:
    print("="*70)
    print("Team:", team)

    # ------------------------------------------------------------
    # (a) All dates when team i played at HOME
    # ------------------------------------------------------------
    home_games = games[games["HomeTeam"] == team].sort_values("Date")
    home_dates = list(home_games["Date"].dt.date.unique())

    print("(a) Home dates:")
    print("   ", home_dates)

    # ------------------------------------------------------------
    # (d) All dates when team i played AWAY
    # ------------------------------------------------------------
    away_games = games[games["AwayTeam"] == team].sort_values("Date")
    away_dates = list(away_games["Date"].dt.date.unique())

    print("(d) Away dates:")
    print("   ", away_dates)

    # ------------------------------------------------------------
    # (b) For each opponent j: number of HOME games vs j
    # ------------------------------------------------------------
    home_vs_count = defaultdict(int)
    for _, row in home_games.iterrows():
        opp = row["AwayTeam"]
        home_vs_count[opp] += 1

    print("(b) Number of HOME games vs each opponent:")
    for opp in sorted(home_vs_count.keys()):
        print(f"     vs {opp}: {home_vs_count[opp]}")

    # ------------------------------------------------------------
    # (c) For each opponent j: number of AWAY games vs j
    # ------------------------------------------------------------
    away_vs_count = defaultdict(int)
    for _, row in away_games.iterrows():
        opp = row["HomeTeam"]
        away_vs_count[opp] += 1

    print("(c) Number of AWAY games @ each opponent:")
    for opp in sorted(away_vs_count.keys()):
        print(f"     @ {opp}: {away_vs_count[opp]}")

print("\nQuestion 1 completed.")


------------------------------------------------------------
Problem 1: Extracting Structural Scheduling Constraints
------------------------------------------------------------

Total number of teams: 16
Team: Atlanta Hawks
(a) Home dates:
    [datetime.date(2025, 11, 3), datetime.date(2025, 11, 7), datetime.date(2025, 11, 15), datetime.date(2025, 11, 17), datetime.date(2025, 11, 19), datetime.date(2025, 11, 23), datetime.date(2025, 11, 27), datetime.date(2025, 11, 28), datetime.date(2025, 11, 29), datetime.date(2025, 12, 25)]
(d) Away dates:
    [datetime.date(2025, 11, 1), datetime.date(2025, 11, 5), datetime.date(2025, 11, 11), datetime.date(2025, 11, 13), datetime.date(2025, 11, 21), datetime.date(2025, 12, 1)]
(b) Number of HOME games vs each opponent:
     vs Boston Celtics: 1
     vs Chicago Bulls: 1
     vs Dallas Mavericks: 1
     vs Golden State Warriors: 1
     vs Los Angeles Lakers: 1
     vs Miami Heat: 1
     vs Milwaukee Bucks: 1
     vs New York Knicks: 1
     vs Phoe

## Problem 2: Integer Programming Model for Feasible Schedules ##

Write an Integer Program (without objective function) whose feasible solutions are all the feasible schedules, where a schedule is feasible if, for each team i:

(e) i plays home (possibly, against a different team) exactly on the dates computed in (a) above; 
(f) plays away (possibly, against a different team) exactly on the dates computed in (d) above; 
(g) for each team j, i plays home against team j exactly the number of times computed in (b) above; (h) for each team j, i plays away against team j (i.e., at j's home) exactly the number of times computed in (c) above.

In [22]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

print("\n------------------------------------------------------------")
print("Problem 2: Integer Programming Model for Feasible Schedules")
print("------------------------------------------------------------\n")

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
path = "games.csv"
games = pd.read_csv(path)
games["Date"] = pd.to_datetime(games["Date"])
teams = sorted(set(games["Home"]).union(games["Visitor"]))
dates = sorted(games["Date"].dt.date.unique())

# ------------------------------------------------------------
# Q1 dictionaries
# ------------------------------------------------------------
home_dates_dict = {
    team: list(games[games["Home"] == team]["Date"].dt.date.unique())
    for team in teams
}

away_dates_dict = {
    team: list(games[games["Visitor"] == team]["Date"].dt.date.unique())
    for team in teams
}

home_vs = {(i, j): 0 for i in teams for j in teams if i != j}
away_vs = {(i, j): 0 for i in teams for j in teams if i != j}

for _, row in games.iterrows():
    i = row["Home"]
    j = row["Visitor"]
    home_vs[(i, j)] += 1
    away_vs[(j, i)] += 1

# ------------------------------------------------------------
# Build Gurobi Model
# ------------------------------------------------------------
m = gp.Model("NBA_Feasible_Schedule")

# Enable solution pool
m.setParam("PoolSearchMode", 2)   
m.setParam("PoolSolutions", 1000)   

# Decision variables
x = m.addVars(teams, teams, dates, vtype=GRB.BINARY, name="x")

# No team plays itself
for i in teams:
    for d in dates:
        m.addConstr(x[i, i, d] == 0)

# Constraint (e): home dates
for i in teams:
    for d in dates:
        if d in home_dates_dict[i]:
            m.addConstr(gp.quicksum(x[i, j, d] for j in teams if j != i) == 1)
        else:
            m.addConstr(gp.quicksum(x[i, j, d] for j in teams if j != i) == 0)

# Constraint (f): away dates
for i in teams:
    for d in dates:
        if d in away_dates_dict[i]:
            m.addConstr(gp.quicksum(x[j, i, d] for j in teams if j != i) == 1)
        else:
            m.addConstr(gp.quicksum(x[j, i, d] for j in teams if j != i) == 0)

# Exactly one game per date per team
for i in teams:
    for d in dates:
        m.addConstr(
            gp.quicksum(x[i, j, d] for j in teams if j != i) +
            gp.quicksum(x[j, i, d] for j in teams if j != i)
            == (1 if ((d in home_dates_dict[i]) or (d in away_dates_dict[i])) else 0)
        )

# Constraint (g): home vs j counts
for i in teams:
    for j in teams:
        if i != j:
            m.addConstr(
                gp.quicksum(x[i, j, d] for d in dates) == home_vs[(i, j)]
            )

# Constraint (h): away vs j counts
for i in teams:
    for j in teams:
        if i != j:
            m.addConstr(
                gp.quicksum(x[j, i, d] for d in dates) == away_vs[(i, j)]
            )

# Feasibility objective
m.setObjective(0, GRB.MINIMIZE)
m.optimize()
print("Model status:", m.status)
if m.status == GRB.OPTIMAL:
    print("Feasible schedule exists!")
    print(f"Number of feasible schedules found: {m.SolCount}")
else:
    print("No feasible schedule exists.")


------------------------------------------------------------
Problem 2: Integer Programming Model for Feasible Schedules
------------------------------------------------------------

Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 1000
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
PoolSolutions  1000
PoolSearchMode  2

Optimize a model with 1504 rows, 4096 columns and 23296 nonzeros
Model fingerprint: 0xd5b11aef
Variable types: 0 continuous, 4096 integer (4096 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 1196 rows and 3654 columns
Presolve time: 0.00s
Presolved: 308 rows, 442 columns, 1386 nonzeros
Variable types: 0 continuous, 442 integer (442 binary)
Found 

In [ ]:
# ------------------------------------------------------------
# Export the feasible schedule found in Question 2 into a CSV file
# Format: Date, Home, Visitor  (same as games.csv)
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Exporting feasible schedule to CSV...")
print("------------------------------------------------------------\n")


schedule_rows = []

# Iterate through Gurobi decision variables
for i in teams:
    for j in teams:
        for d in dates:
            if i != j:
                if x[i, j, d].X > 0.5:     # chosen matchup
                    schedule_rows.append({
                        "Date": pd.to_datetime(d),   # ensure proper format
                        "Home": i,
                        "Visitor": j
                    })

# Convert to DataFrame
schedule_df = pd.DataFrame(schedule_rows)

# Sort by date then home team
schedule_df = schedule_df.sort_values(by=["Date", "Home"]).reset_index(drop=True)

# Save to CSV
output_path = "feasible_schedule.csv"
schedule_df.to_csv(output_path, index=False)

print(f"Feasible schedule exported successfully to: {output_path}")
schedule_df.head()


------------------------------------------------------------
Exporting feasible schedule to CSV...
------------------------------------------------------------

Feasible schedule exported successfully to: feasible_schedule.csv


,Date,Home,Visitor
0,2025-11-01,Boston Celtics,Phoenix Suns
1,2025-11-01,Brooklyn Nets,Denver Nuggets
2,2025-11-01,Chicago Bulls,Atlanta Hawks
3,2025-11-01,Miami Heat,Houston Rockets
4,2025-11-01,Milwaukee Bucks,Dallas Mavericks
